# Model AI phân loại rác sử dụng hình ảnh

# Import Thư Viện

In [2]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms

# Chuẩn bị dữ liệu

In [3]:
from pathlib import Path
DATA_DIR = Path(r"E:\SPCK-CSI18\data\dataset")

TRAIN_DIR = DATA_DIR / "train"
TEST_DIR = DATA_DIR / "test"
VAL_DIR = DATA_DIR / "val"

# Chuẩn hóa dữ liệu

In [4]:
BATCH_SIZE = 32
EPOCHS = 50
LEARNING_RATE = 0.0001
IMG_SIZE = (256, 256)
NUM_CLASSES = 10
WEIGHT_DECAY = 0.05

In [5]:
from torchvision import transforms
from torchvision import datasets


train_transform = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor()
])

val_transform = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.ToTensor()
])

test_transform = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.ToTensor()
])


train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=train_transform)
val_dataset = datasets.ImageFolder(VAL_DIR, transform=val_transform)
test_dataset = datasets.ImageFolder(TEST_DIR, transform=test_transform)


print(train_dataset.classes)
print(len(train_dataset.classes))

['battery', 'biological', 'cardboard', 'clothes', 'glass', 'metal', 'paper', 'plastic', 'shoes', 'trash']
10


# ABCXYZ

In [6]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2
)

In [7]:
images, labels = next(iter(train_loader))

print(images.shape)
print(labels.shape)

torch.Size([32, 3, 256, 256])
torch.Size([32])


# Model

In [8]:
model = models.mobilenet_v3_small(weights="DEFAULT")
print(type(model))

<class 'torchvision.models.mobilenetv3.MobileNetV3'>


In [9]:
model.classifier[-1] = nn.Linear(
    model.classifier[-1].in_features,
    NUM_CLASSES
)
print(model.classifier)
print(model.classifier[-1])

Sequential(
  (0): Linear(in_features=576, out_features=1024, bias=True)
  (1): Hardswish()
  (2): Dropout(p=0.2, inplace=True)
  (3): Linear(in_features=1024, out_features=10, bias=True)
)
Linear(in_features=1024, out_features=10, bias=True)


In [10]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.1,
    patience=3,
    min_lr=1e-6
)

# Training

In [13]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

best_val_acc = 0.0

for epoch in range(EPOCHS):

    # =========================
    # TRAIN
    # =========================
    model.train()

    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        train_loss += loss.item() * images.size(0)

        _, predicted = torch.max(outputs, 1)
        train_total += labels.size(0)
        train_correct += (predicted == labels).sum().item()

    train_loss /= train_total
    train_acc = train_correct / train_total

    # =========================
    # VALIDATION
    # =========================
    model.eval()

    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * images.size(0)

            _, predicted = torch.max(outputs, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

    val_loss /= val_total
    val_acc = val_correct / val_total

    # =========================
    # PRINT
    # =========================
    print(
        f"Epoch {epoch + 1}/{EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.4f}"
    )

    # =========================
    # SAVE LATEST
    # =========================
    torch.save(
        model.state_dict(),
        "garbage_classifier_latest.pth"
    )

    # =========================
    # SAVE BEST
    # =========================
    if val_acc > best_val_acc:

        best_val_acc = val_acc

        torch.save(
            model.state_dict(),
            "garbage_classifier_best.pth"
        )

        print("Best model saved!")

Epoch 1/50 | Train Loss: 0.8821 | Train Acc: 0.7330 | Val Loss: 0.3868 | Val Acc: 0.8786
Best model saved!
Epoch 2/50 | Train Loss: 0.3540 | Train Acc: 0.8890 | Val Loss: 0.2672 | Val Acc: 0.9087
Best model saved!
Epoch 3/50 | Train Loss: 0.2546 | Train Acc: 0.9164 | Val Loss: 0.2209 | Val Acc: 0.9315
Best model saved!
Epoch 4/50 | Train Loss: 0.1843 | Train Acc: 0.9434 | Val Loss: 0.2102 | Val Acc: 0.9405
Best model saved!
Epoch 5/50 | Train Loss: 0.1355 | Train Acc: 0.9559 | Val Loss: 0.1968 | Val Acc: 0.9332
Epoch 6/50 | Train Loss: 0.1072 | Train Acc: 0.9652 | Val Loss: 0.2067 | Val Acc: 0.9356
Epoch 7/50 | Train Loss: 0.0864 | Train Acc: 0.9723 | Val Loss: 0.1914 | Val Acc: 0.9446
Best model saved!
Epoch 8/50 | Train Loss: 0.0619 | Train Acc: 0.9804 | Val Loss: 0.1786 | Val Acc: 0.9438
Epoch 9/50 | Train Loss: 0.0527 | Train Acc: 0.9844 | Val Loss: 0.1889 | Val Acc: 0.9405
Epoch 10/50 | Train Loss: 0.0433 | Train Acc: 0.9869 | Val Loss: 0.2057 | Val Acc: 0.9430
Epoch 11/50 | Train